# Notebook 05 — LLM Explanation Prototype
**Goal:** Prototype FR-05 — sinh giải thích tự nhiên cho alert bằng LLM.

**LLM Provider được hỗ trợ:**
- **Anthropic Claude** (claude-haiku-4-5 — rẻ, nhanh, đủ chất lượng)
- **OpenAI GPT-4o-mini** (fallback)
- Local Llama (stretch)

**Evaluation criteria (PRD §6.4):**
- Groundedness: chỉ dùng thông tin từ triggered rules + top features
- Relevance: đúng pattern AML
- Clarity: tiếng Việt tự nhiên, compliance officer hiểu được

In [1]:
import os
import json
import anthropic
from pathlib import Path
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import List, Dict, Optional

load_dotenv('../.env')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    print('WARNING: ANTHROPIC_API_KEY not set. Set it in .env file.')
else:
    print('Anthropic API key loaded.')

## 1. Alert Data Structure

In [ ]:
@dataclass
class AlertContext:
    transaction_id: str
    decision: str  # FLAG or BLOCK
    risk_score: float
    triggered_rules: List[str]
    rule_reasons: List[str]
    top_features: List[Dict]  # [{name, value, contribution}]
    transaction_summary: Dict  # {type, amount, step}

# Sample alerts for testing
sample_alerts = [
    AlertContext(
        transaction_id='TX-001-STRUCT',
        decision='FLAG',
        risk_score=0.78,
        triggered_rules=['RULE_STRUCTURING', 'RULE_THRESHOLD_AVOIDANCE'],
        rule_reasons=[
            'Structuring: 5 txs in 24h, total 920,000 (above 2x threshold)',
            'Amount 185,000 is 92.5% of CTR threshold 200,000',
        ],
        top_features=[
            {'name': 'tx_count_sender', 'value': 5, 'contribution': 0.31},
            {'name': 'flag_near_threshold', 'value': 1, 'contribution': 0.24},
            {'name': 'amount_vs_avg', 'value': 8.5, 'contribution': 0.18},
        ],
        transaction_summary={'type': 'TRANSFER', 'amount': 185_000, 'step': 72}
    ),
    AlertContext(
        transaction_id='TX-002-DRAIN',
        decision='BLOCK',
        risk_score=0.91,
        triggered_rules=['RULE_ZERO_BALANCE_DRAIN', 'RULE_LARGE_CASHOUT'],
        rule_reasons=[
            'Account fully drained: 500,000 → 0',
            'Large CASH_OUT: 500,000 (above threshold 200,000)',
        ],
        top_features=[
            {'name': 'balance_drain_orig', 'value': 1, 'contribution': 0.42},
            {'name': 'amount_to_balance_ratio', 'value': 1.0, 'contribution': 0.35},
            {'name': 'balance_diff_orig', 'value': 0, 'contribution': 0.08},
        ],
        transaction_summary={'type': 'CASH_OUT', 'amount': 500_000, 'step': 120}
    ),
    AlertContext(
        transaction_id='TX-003-VELOCITY',
        decision='FLAG',
        risk_score=0.65,
        triggered_rules=['RULE_HIGH_VELOCITY'],
        rule_reasons=['Velocity 15.0x baseline (15 txs vs 1.0 baseline)'],
        top_features=[
            {'name': 'tx_count_sender', 'value': 15, 'contribution': 0.45},
            {'name': 'amount_vs_avg', 'value': 3.2, 'contribution': 0.22},
            {'name': 'unique_dest_sender', 'value': 12, 'contribution': 0.18},
        ],
        transaction_summary={'type': 'TRANSFER', 'amount': 50_000, 'step': 200}
    ),
]

print(f'Prepared {len(sample_alerts)} sample alerts for testing')

## 2. Prompt Engineering

In [ ]:
SYSTEM_PROMPT = """Bạn là chuyên gia phân tích AML (Anti-Money Laundering) cho hệ thống ví điện tử.
Nhiệm vụ của bạn là viết giải thích ngắn gọn, rõ ràng cho alert giao dịch đáng ngờ.

Quy tắc bắt buộc:
1. Chỉ sử dụng thông tin được cung cấp — KHÔNG được bịa thêm dữ liệu
2. Viết 2-4 câu bằng tiếng Việt tự nhiên
3. Nêu rõ: pattern AML được phát hiện, lý do cụ thể, và mức độ rủi ro
4. Ngôn ngữ phù hợp cho compliance officer không chuyên về kỹ thuật
5. Không dùng từ kỹ thuật phức tạp như 'SHAP value', 'model feature', etc."""

def build_explanation_prompt(alert: AlertContext) -> str:
    features_text = ', '.join([
        f"{f['name']} = {f['value']} (đóng góp {f['contribution']:.0%})"
        for f in alert.top_features
    ])
    rules_text = '; '.join(alert.rule_reasons)
    
    return f"""Hãy giải thích alert AML sau cho compliance officer:

Thông tin giao dịch:
- Mã giao dịch: {alert.transaction_id}
- Loại: {alert.transaction_summary['type']}
- Số tiền: {alert.transaction_summary['amount']:,.0f} đơn vị
- Quyết định: {alert.decision}
- Điểm rủi ro: {alert.risk_score:.2f}/1.00

Rule đã kích hoạt:
- {rules_text}

Các chỉ số nổi bật:
- {features_text}

Viết giải thích 2-4 câu:"""

# Preview prompt
print(build_explanation_prompt(sample_alerts[0]))

## 3. LLM Explainer Class

In [ ]:
class LLMExplainer:
    def __init__(self, provider: str = 'anthropic', model: str = 'claude-haiku-4-5-20251001'):
        self.provider = provider
        self.model = model
        if provider == 'anthropic':
            self.client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    
    def explain(self, alert: AlertContext) -> str:
        prompt = build_explanation_prompt(alert)
        
        if self.provider == 'anthropic':
            return self._call_anthropic(prompt)
        else:
            return self._mock_explanation(alert)
    
    def _call_anthropic(self, prompt: str) -> str:
        message = self.client.messages.create(
            model=self.model,
            max_tokens=300,
            temperature=0.3,  # low temp for factual, consistent output
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return message.content[0].text
    
    def _mock_explanation(self, alert: AlertContext) -> str:
        """Mock for testing without API key."""
        rule_names = ', '.join(alert.triggered_rules)
        return (
            f"Giao dịch {alert.transaction_id} bị đánh dấu ({alert.decision}) với điểm rủi ro "
            f"{alert.risk_score:.2f} do kích hoạt các rule: {rule_names}. "
            f"{alert.rule_reasons[0] if alert.rule_reasons else ''}. "
            f"Khuyến nghị compliance officer xem xét thêm."
        )

if ANTHROPIC_API_KEY:
    explainer = LLMExplainer(provider='anthropic', model='claude-haiku-4-5-20251001')
    print('Using Anthropic Claude Haiku')
else:
    explainer = LLMExplainer(provider='mock')
    print('Using mock explainer (no API key)')

## 4. Generate Explanations for Sample Alerts

In [ ]:
explanations = []

for alert in sample_alerts:
    print(f'\n{"="*60}')
    print(f'Alert: {alert.transaction_id} | Decision: {alert.decision} | Score: {alert.risk_score:.2f}')
    print(f'Rules: {alert.triggered_rules}')
    print('\n--- LLM Explanation ---')
    
    explanation = explainer.explain(alert)
    print(explanation)
    
    explanations.append({
        'transaction_id': alert.transaction_id,
        'decision': alert.decision,
        'risk_score': alert.risk_score,
        'triggered_rules': alert.triggered_rules,
        'explanation': explanation
    })

## 5. Explanation Quality Evaluation

In [ ]:
# PRD §6.4 qualitative evaluation template
# Mentor sẽ review ≥ 20 alerts theo 3 criteria: Groundedness, Relevance, Clarity

print('=== EXPLANATION QUALITY SCORECARD TEMPLATE ===')
print('(Điền điểm 1-5 cho từng alert, mentor review)\n')

scorecard = []
for i, (alert, exp_data) in enumerate(zip(sample_alerts, explanations)):
    print(f'--- Alert {i+1}: {alert.transaction_id} ---')
    print(f'Explanation: {exp_data["explanation"][:150]}...')
    print('Groundedness [1-5]: ___ (chỉ dùng thông tin có sẵn?)')
    print('Relevance    [1-5]: ___ (đúng pattern AML?)')
    print('Clarity      [1-5]: ___ (dễ hiểu cho compliance officer?)')
    print()

## 6. Caching Strategy (NFR-03: Latency)

In [ ]:
import hashlib

class CachedLLMExplainer(LLMExplainer):
    """LLM Explainer với cache theo (rule_set, top_features) để giảm latency và cost."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cache: Dict[str, str] = {}
    
    def _cache_key(self, alert: AlertContext) -> str:
        key_data = {
            'rules': sorted(alert.triggered_rules),
            'features': sorted([f['name'] for f in alert.top_features]),
            'decision': alert.decision,
            'amount_bucket': int(alert.transaction_summary['amount'] // 50_000),
        }
        return hashlib.md5(json.dumps(key_data, sort_keys=True).encode()).hexdigest()
    
    def explain(self, alert: AlertContext) -> str:
        key = self._cache_key(alert)
        if key in self._cache:
            return f'[CACHED] {self._cache[key]}'
        explanation = super().explain(alert)
        self._cache[key] = explanation
        return explanation

# Test cache behavior
cached_explainer = CachedLLMExplainer(provider='mock')
print('First call:')
print(cached_explainer.explain(sample_alerts[0]))
print('\nSecond call (same pattern — should be cached):')
print(cached_explainer.explain(sample_alerts[0]))
print(f'\nCache size: {len(cached_explainer._cache)} entries')

## 7. Full Alert Output Format (PRD §4.2)

In [ ]:
# Simulate the final API response format from PRD §4.2
import uuid
from datetime import datetime

def format_api_response(alert: AlertContext, explanation: str, latency_ms: float) -> dict:
    return {
        'transaction_id': alert.transaction_id,
        'decision': alert.decision,
        'risk_score': round(alert.risk_score, 4),
        'triggered_rules': alert.triggered_rules,
        'top_features': alert.top_features,
        'explanation': explanation,
        'model_version': 'xgb_aml_v1',
        'latency_ms': latency_ms,
        'timestamp': datetime.utcnow().isoformat() + 'Z',
    }

sample_response = format_api_response(
    sample_alerts[0],
    explanations[0]['explanation'],
    latency_ms=45.2
)

print('=== SAMPLE API RESPONSE ===')
print(json.dumps(sample_response, ensure_ascii=False, indent=2))